# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/17-PythonTkinterSQLiteCRUD.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 17 - Tkinter + SQLite ile CRUD Masaüstü Uygulaması

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bu derste önceki iki dersimizi birleştirerek gerçek bir masaüstü uygulaması geliştireceğiz:

- Tkinter → grafik kullanıcı arayüzü
- SQLite → kalıcı veri saklama
- Python fonksiyonları → uygulama mantığı
- SQL → CRUD işlemleri

> **Çalışma ortamı notu:** Tkinter masaüstü penceresi Google Colab'da normal biçimde açılamaz. Bu notebook ders dokümanı olarak Colab'da incelenebilir; GUI kodları **Visual Studio, VS Code veya yerel Python ortamında** çalıştırılmalıdır.

Bu dersin sonunda öğrencinin:

- SQLite veritabanı oluşturabilmesi,
- Tkinter formu tasarlayabilmesi,
- Treeview ile kayıtları listeleyebilmesi,
- kayıt ekleme,
- kayıt görüntüleme,
- kayıt güncelleme,
- kayıt silme,
- kayıt arama,
- form doğrulama,
- hata yönetimi,
- kullanıcı onayı,
- modüler proje yapısı

gibi özellikleri tek bir uygulamada birleştirebilmesi hedeflenmektedir.


# 1. Projemiz: Öğrenci Kayıt Sistemi

Geliştireceğimiz uygulama bir öğrenci kayıt sistemi olacaktır.

Her öğrenci için şu bilgileri tutacağız:

- ID
- öğrenci adı
- sınıf
- Python puanı
- Matematik puanı

Uygulama şu işlemleri yapabilecek:

- yeni öğrenci ekleme,
- kayıtları listeleme,
- seçilen kaydı forma aktarma,
- kaydı güncelleme,
- kaydı silme,
- isimle arama,
- formu temizleme.


# 2. CRUD Kavramını Hatırlayalım

CRUD dört temel veri işlemini ifade eder:

| İşlem | SQL | Uygulamadaki Karşılığı |
|---|---|---|
| Create | `INSERT` | Yeni kayıt ekleme |
| Read | `SELECT` | Kayıtları listeleme |
| Update | `UPDATE` | Kayıt güncelleme |
| Delete | `DELETE` | Kayıt silme |

Bu dersin ana amacı bu dört işlemi bir GUI uygulamasında kullanmaktır.


# 3. Uygulamanın Veri Akışı

Uygulamamızda veri akışı şu şekilde olacaktır:

```text
Kullanıcı
   ↓
Tkinter Formu
   ↓
Python Fonksiyonları
   ↓
SQL Sorguları
   ↓
SQLite Veritabanı
   ↓
Treeview / Sonuç
```

Arayüz ile veritabanı doğrudan karışık biçimde kullanılmamalıdır. İşlemleri fonksiyonlarla düzenlemek kodun okunabilirliğini artırır.


# 4. Proje Klasörü

İlk aşamada tek dosyalı bir proje geliştirebiliriz:

```text
ogrenci_kayit/
│
├── app.py
└── ogrenciler.db
```

Daha sonra modüler yapıya geçeceğiz:

```text
ogrenci_kayit/
│
├── main.py
├── veritabani.py
├── arayuz.py
└── ogrenciler.db
```


# 5. Kullanacağımız Modüller

Yerel Python dosyamızda:


In [ ]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import sqlite3


Bu modüllerin görevleri:

- `tkinter` → temel arayüz bileşenleri
- `ttk` → Treeview ve daha modern bileşenler
- `messagebox` → bilgi, uyarı ve onay kutuları
- `sqlite3` → veritabanı işlemleri


# 6. Veritabanı Dosya Adı

Dosya adını sabit bir değişkende tutmak kodun yönetimini kolaylaştırır.


In [ ]:
DB = "ogrenciler.db"


Böylece veritabanı adını değiştirmek gerekirse kodun her yerini değiştirmemize gerek kalmaz.


# 7. Veritabanını Hazırlayan Fonksiyon

Uygulama açıldığında tablo yoksa otomatik olarak oluşturalım.


In [ ]:
def veritabani_hazirla():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS Ogrenciler (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            Isim TEXT NOT NULL,
            Sinif INTEGER NOT NULL,
            PythonPuani INTEGER NOT NULL,
            MatematikPuani INTEGER NOT NULL
        )
        """)


Bu fonksiyon uygulama başlarken bir kez çağrılacaktır.


# 8. Veritabanını Başlatmak

In [ ]:
veritabani_hazirla()


# 9. Arayüz Penceresi

Yerel ortamda ana pencereyi oluşturalım.


In [ ]:
pencere = tk.Tk()
pencere.title("Öğrenci Kayıt Sistemi")
pencere.geometry("900x600")


Bu kodu tek başına çalıştırdığınızda pencereyi açık tutmak için dosyanın sonunda:

```python
pencere.mainloop()
```

bulunmalıdır.


# 10. Arayüzü Bölümlere Ayırmak

Arayüzü iki temel alana ayıralım:

- form alanı
- kayıt listesi alanı

Bunun için `Frame` kullanacağız.


In [ ]:
form_frame = tk.Frame(pencere)
form_frame.pack(fill="x", padx=20, pady=15)

liste_frame = tk.Frame(pencere)
liste_frame.pack(fill="both", expand=True, padx=20, pady=10)


# 11. Form Alanı: Öğrenci Adı

In [ ]:
tk.Label(
    form_frame,
    text="Öğrenci Adı"
).grid(
    row=0,
    column=0,
    padx=5,
    pady=5,
    sticky="w"
)

isim_entry = tk.Entry(
    form_frame,
    width=30
)

isim_entry.grid(
    row=0,
    column=1,
    padx=5,
    pady=5
)


# 12. Sınıf Alanı

In [ ]:
tk.Label(
    form_frame,
    text="Sınıf"
).grid(
    row=1,
    column=0,
    padx=5,
    pady=5,
    sticky="w"
)

sinif_entry = tk.Entry(
    form_frame,
    width=30
)

sinif_entry.grid(
    row=1,
    column=1,
    padx=5,
    pady=5
)


# 13. Python Puanı Alanı

In [ ]:
tk.Label(
    form_frame,
    text="Python Puanı"
).grid(
    row=2,
    column=0,
    padx=5,
    pady=5,
    sticky="w"
)

python_entry = tk.Entry(
    form_frame,
    width=30
)

python_entry.grid(
    row=2,
    column=1,
    padx=5,
    pady=5
)


# 14. Matematik Puanı Alanı

In [ ]:
tk.Label(
    form_frame,
    text="Matematik Puanı"
).grid(
    row=3,
    column=0,
    padx=5,
    pady=5,
    sticky="w"
)

matematik_entry = tk.Entry(
    form_frame,
    width=30
)

matematik_entry.grid(
    row=3,
    column=1,
    padx=5,
    pady=5
)


# 15. Seçili Kayıt ID'si

Treeview'dan bir kayıt seçildiğinde hangi veritabanı kaydının seçildiğini bilmemiz gerekir.

Bunu bir değişkende tutacağız.


In [ ]:
secili_id = None


ID kullanıcı tarafından doğrudan girilmeyecek. Veritabanı tarafından otomatik üretilecek.


# 16. Form Temizleme Fonksiyonu

In [ ]:
def formu_temizle():
    global secili_id

    secili_id = None

    isim_entry.delete(0, tk.END)
    sinif_entry.delete(0, tk.END)
    python_entry.delete(0, tk.END)
    matematik_entry.delete(0, tk.END)

    isim_entry.focus()


Form temizlenirken `secili_id` değerini de `None` yapıyoruz. Böylece önceki seçili kayıtla yanlışlıkla işlem yapılmaz.


# 17. Form Doğrulama Fonksiyonu

Veritabanına göndermeden önce kullanıcı girişlerini kontrol edelim.


In [ ]:
def form_verisini_al():
    isim = isim_entry.get().strip()
    sinif_text = sinif_entry.get().strip()
    python_text = python_entry.get().strip()
    matematik_text = matematik_entry.get().strip()

    if not isim or not sinif_text or not python_text or not matematik_text:
        messagebox.showwarning(
            "Eksik Bilgi",
            "Tüm alanları doldurun."
        )
        return None

    try:
        sinif = int(sinif_text)
        python_puani = int(python_text)
        matematik_puani = int(matematik_text)

    except ValueError:
        messagebox.showerror(
            "Hata",
            "Sınıf ve puan alanları sayısal olmalıdır."
        )
        return None

    if python_puani < 0 or python_puani > 100:
        messagebox.showwarning(
            "Geçersiz Puan",
            "Python puanı 0-100 arasında olmalıdır."
        )
        return None

    if matematik_puani < 0 or matematik_puani > 100:
        messagebox.showwarning(
            "Geçersiz Puan",
            "Matematik puanı 0-100 arasında olmalıdır."
        )
        return None

    return isim, sinif, python_puani, matematik_puani


Bu fonksiyon başarılı olursa tuple döndürür:

```python
(isim, sinif, python_puani, matematik_puani)
```

Bir hata varsa `None` döndürür.


# 18. Kayıt Ekleme Fonksiyonu

Create işlemini gerçekleştirelim.


In [ ]:
def kayit_ekle():
    veri = form_verisini_al()

    if veri is None:
        return

    isim, sinif, python_puani, matematik_puani = veri

    try:
        with sqlite3.connect(DB) as baglanti:
            cursor = baglanti.cursor()

            cursor.execute("""
            INSERT INTO Ogrenciler (
                Isim,
                Sinif,
                PythonPuani,
                MatematikPuani
            )
            VALUES (?, ?, ?, ?)
            """, (
                isim,
                sinif,
                python_puani,
                matematik_puani
            ))

        messagebox.showinfo(
            "Başarılı",
            "Öğrenci kaydedildi."
        )

        formu_temizle()
        kayitlari_listele()

    except sqlite3.Error as hata:
        messagebox.showerror(
            "Veritabanı Hatası",
            str(hata)
        )


Burada önemli üç işlem birlikte yapılır:

1. veri doğrulanır,
2. SQLite'a eklenir,
3. Treeview yeniden yüklenir.


# 19. Treeview Nedir?

`ttk.Treeview`, tablo biçimindeki verileri arayüzde göstermek için kullanılır.

Öğrenci kayıtlarımız için şu sütunları göstereceğiz:

- ID
- İsim
- Sınıf
- Python
- Matematik
- Ortalama


# 20. Treeview Oluşturmak

In [ ]:
sutunlar = (
    "Id",
    "Isim",
    "Sinif",
    "Python",
    "Matematik",
    "Ortalama"
)

tablo = ttk.Treeview(
    liste_frame,
    columns=sutunlar,
    show="headings"
)


# 21. Treeview Başlıklarını Ayarlamak

In [ ]:
tablo.heading("Id", text="ID")
tablo.heading("Isim", text="Öğrenci Adı")
tablo.heading("Sinif", text="Sınıf")
tablo.heading("Python", text="Python")
tablo.heading("Matematik", text="Matematik")
tablo.heading("Ortalama", text="Ortalama")


# 22. Sütun Genişliklerini Ayarlamak

In [ ]:
tablo.column("Id", width=60, anchor="center")
tablo.column("Isim", width=180)
tablo.column("Sinif", width=80, anchor="center")
tablo.column("Python", width=100, anchor="center")
tablo.column("Matematik", width=100, anchor="center")
tablo.column("Ortalama", width=100, anchor="center")


# 23. Treeview'ı Yerleştirmek

In [ ]:
tablo.pack(
    fill="both",
    expand=True
)


# 24. Kayıtları Veritabanından Okumak

Read işlemi:


In [ ]:
def kayitlari_getir():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT
            Id,
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani,
            (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
        FROM Ogrenciler
        ORDER BY Id DESC
        """)

        return cursor.fetchall()


# 25. Treeview İçeriğini Temizlemek

In [ ]:
def tabloyu_temizle():
    for satir in tablo.get_children():
        tablo.delete(satir)


Treeview yeniden doldurulmadan önce eski satırları temizlemek gerekir.


# 26. Kayıtları Treeview'da Listelemek

In [ ]:
def kayitlari_listele():
    tabloyu_temizle()

    for kayit in kayitlari_getir():
        kayit = list(kayit)
        kayit[5] = round(kayit[5], 2)

        tablo.insert(
            "",
            tk.END,
            values=kayit
        )


# 27. Uygulama Açılırken Listeleme

Pencere açılmadan önce:


In [ ]:
kayitlari_listele()


# 28. Treeview Seçim Olayı

Kullanıcı bir satıra tıkladığında kayıt bilgilerini forma aktarmak istiyoruz.

Bunun için `<<TreeviewSelect>>` olayını kullanabiliriz.


# 29. Seçilen Kaydı Okumak

In [ ]:
def kayit_secildi(event):
    global secili_id

    secim = tablo.selection()

    if not secim:
        return

    item = tablo.item(secim[0])
    degerler = item["values"]

    secili_id = degerler[0]

    isim_entry.delete(0, tk.END)
    isim_entry.insert(0, degerler[1])

    sinif_entry.delete(0, tk.END)
    sinif_entry.insert(0, degerler[2])

    python_entry.delete(0, tk.END)
    python_entry.insert(0, degerler[3])

    matematik_entry.delete(0, tk.END)
    matematik_entry.insert(0, degerler[4])


# 30. Treeview Seçim Olayını Bağlamak

In [ ]:
tablo.bind(
    "<<TreeviewSelect>>",
    kayit_secildi
)


Artık kullanıcı satıra tıkladığında bilgiler Entry alanlarına gelir.


# 31. Güncelleme İşlemi

Update işlemi için önce bir kayıt seçilmiş olmalıdır.


In [ ]:
def kayit_guncelle():
    if secili_id is None:
        messagebox.showwarning(
            "Kayıt Seçilmedi",
            "Güncellemek için önce bir kayıt seçin."
        )
        return

    veri = form_verisini_al()

    if veri is None:
        return

    isim, sinif, python_puani, matematik_puani = veri

    try:
        with sqlite3.connect(DB) as baglanti:
            cursor = baglanti.cursor()

            cursor.execute("""
            UPDATE Ogrenciler
            SET
                Isim = ?,
                Sinif = ?,
                PythonPuani = ?,
                MatematikPuani = ?
            WHERE Id = ?
            """, (
                isim,
                sinif,
                python_puani,
                matematik_puani,
                secili_id
            ))

        messagebox.showinfo(
            "Başarılı",
            "Kayıt güncellendi."
        )

        formu_temizle()
        kayitlari_listele()

    except sqlite3.Error as hata:
        messagebox.showerror(
            "Veritabanı Hatası",
            str(hata)
        )


# 32. Silme İşlemi

Delete işlemi geri alınması zor bir işlem olduğu için kullanıcıdan onay alalım.


In [ ]:
def kayit_sil():
    global secili_id

    if secili_id is None:
        messagebox.showwarning(
            "Kayıt Seçilmedi",
            "Silmek için önce bir kayıt seçin."
        )
        return

    cevap = messagebox.askyesno(
        "Silme Onayı",
        "Seçili öğrenci kaydı silinsin mi?"
    )

    if not cevap:
        return

    try:
        with sqlite3.connect(DB) as baglanti:
            cursor = baglanti.cursor()

            cursor.execute(
                "DELETE FROM Ogrenciler WHERE Id = ?",
                (secili_id,)
            )

        messagebox.showinfo(
            "Başarılı",
            "Kayıt silindi."
        )

        formu_temizle()
        kayitlari_listele()

    except sqlite3.Error as hata:
        messagebox.showerror(
            "Veritabanı Hatası",
            str(hata)
        )


# 33. Neden ID ile Silme Yapıyoruz?

İsim aynı olabilir.

Örneğin iki farklı öğrenci:

```text
Ali
Ali
```

adını taşıyabilir.

Bu nedenle kayıt güncelleme ve silme işlemlerinde benzersiz olan `Id` alanını kullanmak daha doğrudur.


# 34. Arama Alanı Oluşturmak

In [ ]:
arama_frame = tk.Frame(pencere)
arama_frame.pack(
    fill="x",
    padx=20,
    pady=5
)

tk.Label(
    arama_frame,
    text="Öğrenci Ara:"
).pack(side="left")

arama_entry = tk.Entry(
    arama_frame,
    width=30
)
arama_entry.pack(
    side="left",
    padx=5
)


# 35. İsimle Arama Fonksiyonu

In [ ]:
def kayit_ara():
    kelime = arama_entry.get().strip()

    tabloyu_temizle()

    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT
            Id,
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani,
            (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
        FROM Ogrenciler
        WHERE Isim LIKE ?
        ORDER BY Isim
        """, (
            f"%{kelime}%",
        ))

        kayitlar = cursor.fetchall()

    for kayit in kayitlar:
        kayit = list(kayit)
        kayit[5] = round(kayit[5], 2)

        tablo.insert(
            "",
            tk.END,
            values=kayit
        )


# 36. Aramayı Temizlemek

In [ ]:
def aramayi_temizle():
    arama_entry.delete(0, tk.END)
    kayitlari_listele()


# 37. Buton Alanı

In [ ]:
buton_frame = tk.Frame(
    form_frame
)

buton_frame.grid(
    row=4,
    column=0,
    columnspan=2,
    pady=15
)


# 38. Ekle Butonu

In [ ]:
tk.Button(
    buton_frame,
    text="Ekle",
    width=12,
    command=kayit_ekle
).grid(
    row=0,
    column=0,
    padx=5
)


# 39. Güncelle Butonu

In [ ]:
tk.Button(
    buton_frame,
    text="Güncelle",
    width=12,
    command=kayit_guncelle
).grid(
    row=0,
    column=1,
    padx=5
)


# 40. Sil Butonu

In [ ]:
tk.Button(
    buton_frame,
    text="Sil",
    width=12,
    command=kayit_sil
).grid(
    row=0,
    column=2,
    padx=5
)


# 41. Temizle Butonu

In [ ]:
tk.Button(
    buton_frame,
    text="Temizle",
    width=12,
    command=formu_temizle
).grid(
    row=0,
    column=3,
    padx=5
)


# 42. Ara ve Tümünü Göster Butonları

In [ ]:
tk.Button(
    arama_frame,
    text="Ara",
    command=kayit_ara
).pack(
    side="left",
    padx=5
)

tk.Button(
    arama_frame,
    text="Tümünü Göster",
    command=aramayi_temizle
).pack(
    side="left",
    padx=5
)


# 43. Enter ile Arama

Kullanıcı arama alanındayken Enter'a basınca arama yapılabilir.


In [ ]:
arama_entry.bind(
    "<Return>",
    lambda event: kayit_ara()
)


# 44. Scrollbar Eklemek

Kayıt sayısı arttığında Treeview için kaydırma çubuğu gerekir.


In [ ]:
scrollbar = ttk.Scrollbar(
    liste_frame,
    orient="vertical",
    command=tablo.yview
)

tablo.configure(
    yscrollcommand=scrollbar.set
)


Gerçek projede `Treeview` ve scrollbar'ı aynı frame içinde `grid()` kullanarak daha düzenli yerleştirebiliriz.


# 45. Durum Bilgisi Göstermek

Kullanıcıya toplam kayıt sayısını gösterebiliriz.


In [ ]:
durum_label = tk.Label(
    pencere,
    text=""
)

durum_label.pack(
    pady=5
)


# 46. Kayıt Sayısını Hesaplamak

In [ ]:
def kayit_sayisi():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute(
            "SELECT COUNT(*) FROM Ogrenciler"
        )

        return cursor.fetchone()[0]


# 47. Listeleme Fonksiyonunu Geliştirmek

Listeleme sonunda durum bilgisini de güncelleyebiliriz.


In [ ]:
def kayitlari_listele():
    tabloyu_temizle()

    kayitlar = kayitlari_getir()

    for kayit in kayitlar:
        kayit = list(kayit)
        kayit[5] = round(kayit[5], 2)

        tablo.insert(
            "",
            tk.END,
            values=kayit
        )

    durum_label.config(
        text=f"Toplam kayıt: {len(kayitlar)}"
    )


# 48. Pencere Kapanırken Onay

Kullanıcı pencereyi kapatırken onay sorabiliriz.


In [ ]:
def cikis():
    cevap = messagebox.askyesno(
        "Çıkış",
        "Programdan çıkmak istiyor musunuz?"
    )

    if cevap:
        pencere.destroy()


# 49. Pencere Kapatma Olayını Yakalamak

In [ ]:
pencere.protocol(
    "WM_DELETE_WINDOW",
    cikis
)


# 50. Menü Eklemek

Basit bir Dosya menüsü ekleyebiliriz.


In [ ]:
menu_cubugu = tk.Menu(pencere)

dosya_menu = tk.Menu(
    menu_cubugu,
    tearoff=0
)

dosya_menu.add_command(
    label="Yeni Kayıt",
    command=formu_temizle
)

dosya_menu.add_separator()

dosya_menu.add_command(
    label="Çıkış",
    command=cikis
)

menu_cubugu.add_cascade(
    label="Dosya",
    menu=dosya_menu
)

pencere.config(
    menu=menu_cubugu
)


# 51. Uygulamayı Başlatmak

Dosyanın sonunda:


In [ ]:
pencere.mainloop()


Bu satır Tkinter olay döngüsünü başlatır.

Yerel Python ortamında çalıştırıldığında uygulama penceresi açılır.


# 52. Tam Uygulama Akışı

Artık uygulamamızın işleyişi şöyledir:

### Program başlar
`veritabani_hazirla()`

### Kayıtlar yüklenir
`kayitlari_listele()`

### Kullanıcı veri girer
Entry alanları

### Ekle
`INSERT`

### Satır seç
Treeview → form

### Güncelle
`UPDATE`

### Sil
`DELETE`

### Ara
`SELECT ... WHERE Isim LIKE ?`

### Listele
`SELECT`

Bu yapı gerçek bir CRUD uygulamasıdır.


# 53. Tam Tek Dosyalık Uygulama

Aşağıdaki örnek tüm temel parçaları birleştiren tek dosyalık sürümdür.

Bu kodu `app.py` olarak kaydedip Visual Studio veya VS Code üzerinde çalıştırabilirsiniz.


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import sqlite3

DB = "ogrenciler.db"

def veritabani_hazirla():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS Ogrenciler (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            Isim TEXT NOT NULL,
            Sinif INTEGER NOT NULL,
            PythonPuani INTEGER NOT NULL,
            MatematikPuani INTEGER NOT NULL
        )
        """)

def form_verisini_al():
    isim = isim_entry.get().strip()
    sinif_text = sinif_entry.get().strip()
    python_text = python_entry.get().strip()
    matematik_text = matematik_entry.get().strip()

    if not isim or not sinif_text or not python_text or not matematik_text:
        messagebox.showwarning("Eksik Bilgi", "Tüm alanları doldurun.")
        return None

    try:
        sinif = int(sinif_text)
        python_puani = int(python_text)
        matematik_puani = int(matematik_text)
    except ValueError:
        messagebox.showerror("Hata", "Sınıf ve puanlar sayısal olmalıdır.")
        return None

    if not 0 <= python_puani <= 100:
        messagebox.showwarning("Geçersiz Puan", "Python puanı 0-100 arasında olmalıdır.")
        return None

    if not 0 <= matematik_puani <= 100:
        messagebox.showwarning("Geçersiz Puan", "Matematik puanı 0-100 arasında olmalıdır.")
        return None

    return isim, sinif, python_puani, matematik_puani

def formu_temizle():
    global secili_id

    secili_id = None

    for entry in (isim_entry, sinif_entry, python_entry, matematik_entry):
        entry.delete(0, tk.END)

    isim_entry.focus()

def tabloyu_temizle():
    for satir in tablo.get_children():
        tablo.delete(satir)

def kayitlari_getir(kelime=None):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        if kelime:
            cursor.execute("""
            SELECT
                Id,
                Isim,
                Sinif,
                PythonPuani,
                MatematikPuani,
                (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
            FROM Ogrenciler
            WHERE Isim LIKE ?
            ORDER BY Isim
            """, (f"%{kelime}%",))
        else:
            cursor.execute("""
            SELECT
                Id,
                Isim,
                Sinif,
                PythonPuani,
                MatematikPuani,
                (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
            FROM Ogrenciler
            ORDER BY Id DESC
            """)

        return cursor.fetchall()

def kayitlari_listele(kelime=None):
    tabloyu_temizle()

    kayitlar = kayitlari_getir(kelime)

    for kayit in kayitlar:
        degerler = list(kayit)
        degerler[5] = round(degerler[5], 2)

        tablo.insert("", tk.END, values=degerler)

    durum_label.config(text=f"Toplam kayıt: {len(kayitlar)}")

def kayit_ekle():
    veri = form_verisini_al()

    if veri is None:
        return

    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        INSERT INTO Ogrenciler (
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani
        )
        VALUES (?, ?, ?, ?)
        """, veri)

    messagebox.showinfo("Başarılı", "Öğrenci kaydedildi.")

    formu_temizle()
    kayitlari_listele()

def kayit_guncelle():
    if secili_id is None:
        messagebox.showwarning("Kayıt Seçilmedi", "Önce bir kayıt seçin.")
        return

    veri = form_verisini_al()

    if veri is None:
        return

    isim, sinif, python_puani, matematik_puani = veri

    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        UPDATE Ogrenciler
        SET
            Isim = ?,
            Sinif = ?,
            PythonPuani = ?,
            MatematikPuani = ?
        WHERE Id = ?
        """, (
            isim,
            sinif,
            python_puani,
            matematik_puani,
            secili_id
        ))

    messagebox.showinfo("Başarılı", "Kayıt güncellendi.")

    formu_temizle()
    kayitlari_listele()

def kayit_sil():
    if secili_id is None:
        messagebox.showwarning("Kayıt Seçilmedi", "Önce bir kayıt seçin.")
        return

    cevap = messagebox.askyesno(
        "Silme Onayı",
        "Seçili kayıt silinsin mi?"
    )

    if not cevap:
        return

    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute(
            "DELETE FROM Ogrenciler WHERE Id = ?",
            (secili_id,)
        )

    messagebox.showinfo("Başarılı", "Kayıt silindi.")

    formu_temizle()
    kayitlari_listele()

def kayit_secildi(event):
    global secili_id

    secim = tablo.selection()

    if not secim:
        return

    degerler = tablo.item(secim[0])["values"]

    secili_id = degerler[0]

    entryler = (
        (isim_entry, degerler[1]),
        (sinif_entry, degerler[2]),
        (python_entry, degerler[3]),
        (matematik_entry, degerler[4])
    )

    for entry, deger in entryler:
        entry.delete(0, tk.END)
        entry.insert(0, deger)

def ara():
    kelime = arama_entry.get().strip()
    kayitlari_listele(kelime)

def tumunu_goster():
    arama_entry.delete(0, tk.END)
    kayitlari_listele()

def cikis():
    if messagebox.askyesno("Çıkış", "Programdan çıkmak istiyor musunuz?"):
        pencere.destroy()

veritabani_hazirla()

pencere = tk.Tk()
pencere.title("Öğrenci Kayıt Sistemi")
pencere.geometry("900x600")

secili_id = None

form_frame = tk.Frame(pencere)
form_frame.pack(fill="x", padx=20, pady=15)

alanlar = [
    ("Öğrenci Adı", 0),
    ("Sınıf", 1),
    ("Python Puanı", 2),
    ("Matematik Puanı", 3)
]

for metin, satir in alanlar:
    tk.Label(
        form_frame,
        text=metin
    ).grid(
        row=satir,
        column=0,
        padx=5,
        pady=5,
        sticky="w"
    )

isim_entry = tk.Entry(form_frame, width=30)
sinif_entry = tk.Entry(form_frame, width=30)
python_entry = tk.Entry(form_frame, width=30)
matematik_entry = tk.Entry(form_frame, width=30)

isim_entry.grid(row=0, column=1, padx=5, pady=5)
sinif_entry.grid(row=1, column=1, padx=5, pady=5)
python_entry.grid(row=2, column=1, padx=5, pady=5)
matematik_entry.grid(row=3, column=1, padx=5, pady=5)

buton_frame = tk.Frame(form_frame)
buton_frame.grid(row=4, column=0, columnspan=2, pady=15)

tk.Button(buton_frame, text="Ekle", width=12, command=kayit_ekle).grid(row=0, column=0, padx=5)
tk.Button(buton_frame, text="Güncelle", width=12, command=kayit_guncelle).grid(row=0, column=1, padx=5)
tk.Button(buton_frame, text="Sil", width=12, command=kayit_sil).grid(row=0, column=2, padx=5)
tk.Button(buton_frame, text="Temizle", width=12, command=formu_temizle).grid(row=0, column=3, padx=5)

arama_frame = tk.Frame(pencere)
arama_frame.pack(fill="x", padx=20, pady=5)

tk.Label(arama_frame, text="Öğrenci Ara:").pack(side="left")

arama_entry = tk.Entry(arama_frame, width=30)
arama_entry.pack(side="left", padx=5)

tk.Button(arama_frame, text="Ara", command=ara).pack(side="left", padx=5)
tk.Button(arama_frame, text="Tümünü Göster", command=tumunu_goster).pack(side="left", padx=5)

liste_frame = tk.Frame(pencere)
liste_frame.pack(fill="both", expand=True, padx=20, pady=10)

sutunlar = ("Id", "Isim", "Sinif", "Python", "Matematik", "Ortalama")

tablo = ttk.Treeview(
    liste_frame,
    columns=sutunlar,
    show="headings"
)

basliklar = {
    "Id": "ID",
    "Isim": "Öğrenci Adı",
    "Sinif": "Sınıf",
    "Python": "Python",
    "Matematik": "Matematik",
    "Ortalama": "Ortalama"
}

for sutun, baslik in basliklar.items():
    tablo.heading(sutun, text=baslik)

tablo.column("Id", width=60, anchor="center")
tablo.column("Isim", width=180)
tablo.column("Sinif", width=80, anchor="center")
tablo.column("Python", width=100, anchor="center")
tablo.column("Matematik", width=100, anchor="center")
tablo.column("Ortalama", width=100, anchor="center")

scrollbar = ttk.Scrollbar(
    liste_frame,
    orient="vertical",
    command=tablo.yview
)

tablo.configure(yscrollcommand=scrollbar.set)

tablo.pack(side="left", fill="both", expand=True)
scrollbar.pack(side="right", fill="y")

tablo.bind("<<TreeviewSelect>>", kayit_secildi)
arama_entry.bind("<Return>", lambda event: ara())

durum_label = tk.Label(pencere, text="")
durum_label.pack(pady=5)

pencere.protocol("WM_DELETE_WINDOW", cikis)

kayitlari_listele()

pencere.mainloop()


# 54. Tam Uygulamadaki Önemli Noktalar

Tam uygulamada özellikle şu yapılar birleşmiştir:

- `sqlite3.connect()`
- `CREATE TABLE`
- `INSERT`
- `SELECT`
- `UPDATE`
- `DELETE`
- parametreli SQL sorguları
- `Entry`
- `Button`
- `Frame`
- `Treeview`
- `Scrollbar`
- `messagebox`
- `bind()`
- form doğrulama
- hata önleme
- ID yönetimi

Bu, şimdiye kadar öğrendiğimiz Python konularının gerçek bir uygulamada birleştiği önemli bir aşamadır.


# 55. Uygulamayı Modüllere Ayırmak

Tek dosya öğrenmek için uygundur. Ancak proje büyüdükçe kodu bölmek gerekir.

Örnek:

```text
ogrenci_kayit/
│
├── main.py
├── veritabani.py
└── ogrenciler.db
```

`veritabani.py`:

- tablo oluşturma,
- ekleme,
- listeleme,
- güncelleme,
- silme,
- arama

işlemlerini içerir.

`main.py`:

- Tkinter arayüzünü,
- kullanıcı olaylarını,
- form işlemlerini

içerir.


# 56. `veritabani.py` Örneği

In [ ]:
import sqlite3

DB = "ogrenciler.db"

def hazirla():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS Ogrenciler (
            Id INTEGER PRIMARY KEY AUTOINCREMENT,
            Isim TEXT NOT NULL,
            Sinif INTEGER NOT NULL,
            PythonPuani INTEGER NOT NULL,
            MatematikPuani INTEGER NOT NULL
        )
        """)

def ekle(isim, sinif, python_puani, matematik_puani):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        INSERT INTO Ogrenciler (
            Isim, Sinif, PythonPuani, MatematikPuani
        )
        VALUES (?, ?, ?, ?)
        """, (isim, sinif, python_puani, matematik_puani))

def listele():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        SELECT
            Id,
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani,
            (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
        FROM Ogrenciler
        ORDER BY Id DESC
        """)
        return cursor.fetchall()

def guncelle(ogrenci_id, isim, sinif, python_puani, matematik_puani):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        UPDATE Ogrenciler
        SET
            Isim = ?,
            Sinif = ?,
            PythonPuani = ?,
            MatematikPuani = ?
        WHERE Id = ?
        """, (isim, sinif, python_puani, matematik_puani, ogrenci_id))

def sil(ogrenci_id):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute(
            "DELETE FROM Ogrenciler WHERE Id = ?",
            (ogrenci_id,)
        )

def ara(kelime):
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()
        cursor.execute("""
        SELECT
            Id,
            Isim,
            Sinif,
            PythonPuani,
            MatematikPuani,
            (PythonPuani + MatematikPuani) / 2.0 AS Ortalama
        FROM Ogrenciler
        WHERE Isim LIKE ?
        ORDER BY Isim
        """, (f"%{kelime}%",))
        return cursor.fetchall()


# 57. `main.py` İçinden Modülü Kullanmak

Örneğin:


In [ ]:
import veritabani

veritabani.hazirla()

kayitlar = veritabani.listele()

for kayit in kayitlar:
    print(kayit)


Bu yaklaşım kodun görevlerini ayırır ve uygulamanın bakımını kolaylaştırır.


# 58. İleri Geliştirme: Öğrenci Numarası

Uygulamaya `OgrenciNo` alanı eklenebilir.

Bu alan tekrar etmemeli:

```sql
OgrenciNo INTEGER UNIQUE NOT NULL
```

Böylece aynı öğrenci numarası iki kez kaydedilemez.


# 59. `IntegrityError` Yakalamak

UNIQUE kuralına aykırı kayıt eklenirse `sqlite3.IntegrityError` oluşabilir.

Örnek:

```python
try:
    ...
except sqlite3.IntegrityError:
    messagebox.showerror(
        "Hata",
        "Bu öğrenci numarası zaten kayıtlı."
    )
```


# 60. İleri Geliştirme: Sınıfı Combobox Yapmak

Sınıf girişini serbest Entry yerine Combobox yapabiliriz.

```python
sinif_combo = ttk.Combobox(
    form_frame,
    values=[5, 6, 7, 8],
    state="readonly"
)
```

Bu yaklaşım yanlış veri girişini azaltır.


# 61. İleri Geliştirme: Ortalama Sütunu

Ortalama değeri veritabanında ayrıca saklamak zorunda değiliz.

Şu sorguyla hesaplayabiliriz:

```sql
(PythonPuani + MatematikPuani) / 2.0 AS Ortalama
```

Türetilmiş değerleri gerektiğinde hesaplamak veri tekrarını azaltabilir.


# 62. İleri Geliştirme: Başarı Durumu

SQL veya Python ile:

```text
Ortalama >= 50 → Başarılı
Ortalama < 50 → Geliştirilmeli
```

gibi bir durum oluşturabiliriz.


# 63. İleri Geliştirme: İstatistik Paneli

Arayüzde şu bilgileri gösterebiliriz:

- toplam öğrenci sayısı,
- genel ortalama,
- en yüksek puan,
- en düşük puan,
- 80 ve üzeri öğrenci sayısı.

Bunlar SQLite sorguları ile hesaplanabilir.


# 64. Örnek İstatistik Sorgusu

In [ ]:
def istatistikler():
    with sqlite3.connect(DB) as baglanti:
        cursor = baglanti.cursor()

        cursor.execute("""
        SELECT
            COUNT(*),
            AVG((PythonPuani + MatematikPuani) / 2.0),
            MAX((PythonPuani + MatematikPuani) / 2.0),
            MIN((PythonPuani + MatematikPuani) / 2.0)
        FROM Ogrenciler
        """)

        return cursor.fetchone()


# 65. İleri Geliştirme: Pandas Raporu

SQLite verilerini Pandas'a aktararak daha gelişmiş raporlar oluşturabiliriz.

```python
with sqlite3.connect(DB) as baglanti:
    df = pd.read_sql_query(
        "SELECT * FROM Ogrenciler",
        baglanti
    )
```

Ardından:

- ortalama,
- filtreleme,
- gruplama,
- CSV dışa aktarma,
- grafik

işlemleri yapılabilir.


# 66. İleri Geliştirme: CSV Dışa Aktarma

Kullanıcıya verileri CSV olarak dışarı aktarma butonu eklenebilir.

```python
df.to_csv(
    "ogrenci_raporu.csv",
    index=False,
    encoding="utf-8"
)
```

Bu, masaüstü uygulamasını veri analizi derslerimizle birleştirir.


# 67. İleri Geliştirme: Grafik Butonu

Bir butona basıldığında öğrenci ortalamalarının Matplotlib grafiği gösterilebilir.

Akış:

```text
SQLite
↓
Pandas
↓
Matplotlib
↓
Grafik
```

Bu özellik masaüstü uygulamasını raporlama aracına dönüştürebilir.


# 68. Masaüstü Uygulamalarında Kullanıcı Deneyimi

Çalışan kod tek başına yeterli değildir.

İyi bir uygulamada:

- buton isimleri açık olmalı,
- hata mesajları anlaşılır olmalı,
- boş alanlar kontrol edilmeli,
- silme işleminde onay alınmalı,
- işlem sonrası form temizlenmeli,
- tablo otomatik yenilenmeli,
- kullanıcı yanlış işlem yapmaya karşı yönlendirilmelidir.


# 69. Güvenli Veri İşlemleri

Veritabanı kodlarında:

- parametreli SQL kullanın,
- kullanıcı girdilerini doğrulayın,
- `WHERE` olmadan `DELETE` veya `UPDATE` kullanırken çok dikkatli olun,
- işlem sonrası kullanıcıya bilgi verin,
- silme işlemlerinde onay alın,
- benzersiz kayıtları ID ile yönetin.

Bu alışkanlıklar ileride web ve daha büyük veritabanı projelerinde de önemini korur.


# 70. Ders Özeti

Bu derste:

- Tkinter + SQLite entegrasyonu,
- CRUD,
- form tasarımı,
- Frame,
- Entry,
- Button,
- Treeview,
- Scrollbar,
- seçili kayıt yönetimi,
- kayıt ekleme,
- kayıt listeleme,
- kayıt seçme,
- kayıt güncelleme,
- kayıt silme,
- silme onayı,
- arama,
- Enter ile arama,
- toplam kayıt göstergesi,
- çıkış onayı,
- form doğrulama,
- parametreli SQL,
- modüler proje yapısı,
- veri tabanı katmanı,
- istatistik ve raporlama geliştirmeleri

konularını bir proje üzerinde birleştirdik.


# 71. Mini Uygulamalar

1. Öğrenci formuna `OgrenciNo` alanı ekleyin.
2. Öğrenci numarasını `UNIQUE` yapın.
3. Aynı öğrenci numarası girildiğinde hata mesajı gösterin.
4. Sınıf alanını Combobox yapın.
5. Öğrenci adına göre arama yapın.
6. Öğrenci numarasına göre arama özelliği ekleyin.
7. Python puanına göre büyükten küçüğe sıralama butonu ekleyin.
8. Genel ortalamaya göre sıralama ekleyin.
9. Treeview'a başarı durumu sütunu ekleyin.
10. Ortalama 50 altındaki öğrencileri listeleyen buton ekleyin.
11. 80 ve üzeri öğrencileri filtreleyen buton ekleyin.
12. Toplam öğrenci sayısını arayüzde gösterin.
13. Genel başarı ortalamasını arayüzde gösterin.
14. En yüksek puanı gösterin.
15. En düşük puanı gösterin.
16. CSV dışa aktarma butonu ekleyin.
17. Pandas kullanarak sınıf ortalaması raporu üretin.
18. Matplotlib ile öğrenci ortalama grafiği oluşturun.
19. Veritabanı yedekleme butonu ekleyin.
20. Kayıt silmeden önce öğrenci adını onay mesajında gösterin.
21. Form alanlarına klavye kısayolları ekleyin.
22. Treeview çift tıklamasında düzenleme formu açın.
23. Uygulamayı `main.py` ve `veritabani.py` olarak iki modüle ayırın.
24. Kendi stok takip uygulamanızı aynı CRUD mantığıyla geliştirin.
25. Uygulamayı gerçek kullanıcı testi yapabilecek hale getirin.


# 72. Proje Görevi

Dersi tamamlamak için **tam bir masaüstü kayıt uygulaması** geliştirin.

Projelerden biri seçilebilir:

- Öğrenci kayıt sistemi
- Stok takip sistemi
- Kütüphane kitap sistemi
- Harcama takip sistemi
- Etkinlik kayıt sistemi

Projede en az şu özellikler bulunsun:

- SQLite veritabanı,
- en az 5 veri alanı,
- `INSERT`,
- `SELECT`,
- `UPDATE`,
- `DELETE`,
- Treeview,
- kayıt arama,
- kayıt filtreleme,
- form doğrulama,
- hata yönetimi,
- silme onayı,
- otomatik liste yenileme,
- en az bir istatistik,
- en az bir raporlama özelliği.

Kod mümkünse iki veya daha fazla modüle ayrılmalıdır.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki tam uygulama zincirini kurabilmesi hedeflenmektedir:

**Kullanıcı**

↓

**Tkinter Arayüzü**

↓

**Olaylar ve Fonksiyonlar**

↓

**SQL CRUD İşlemleri**

↓

**SQLite Veritabanı**

↓

**Treeview / Rapor / Sonuç**

Bu aşamada öğrenciler yalnızca Python komutları yazmıyor; gerçek bir **masaüstü bilgi sistemi** geliştirmeye başlamış durumdadır.

Bundan sonraki aşamada web uygulamalarına geçerek aynı CRUD ve veritabanı mantığının tarayıcı üzerinden nasıl kullanılacağını öğrenebiliriz.
